# DETECTOR DE FUNDAMENTALES Y ARMÓNICOS

La idea consiste en tomar el micrófono de referencia que tiene la mejor toma con mejor SNR de cada grabación por cada giro de la mesa giratoria. A esa grabación procesarla y encontrar las frecuencias fundamentales que se tienen y en que instante suceden.

## Levantar directorio de audios

In [26]:
import librosa          # audio, F0, STFT
import numpy as np      # matrices y cálculos
import soundfile as sf  # leer WAVs (más rápido que librosa.load)
import matplotlib.pyplot as plt  # plots
import ipywidgets       # sliders interactivos en el notebook
import soundfile as sf
import numpy as np
from pathlib import Path
import re

### Directorio y formato de archivos

In [27]:
# ── Configuración ──────────────────────────────────────────────────────────────
PATH = r'D:\UNTREF\IMA\TP5 - PATRON POLAR\procesados'

# Regex para extraer datos de los nombres de los archivos de micrófonos, referencia y calibración

PLANTILLA_MICS = r'mic_(?P<MIC>\d+)_ang_(?P<DIN>\w+)_(?P<ANG>\d+)\.wav'
PLANTILLA_REF  = r'mic_ref_ang_(?P<DIN>\w+)_(?P<ANG>\d+)_alineado\.wav'
PLANTILLA_CAL  = r'mic_(?P<MIC>\d+)_ang_cal\.wav'

# ── Rutas ──────────────────────────────────────────────────────────────────────
carpeta_base  = Path(PATH)
carpeta_media = carpeta_base / 'media'
carpeta_ref   = carpeta_base / 'referencia'
carpeta_cal   = carpeta_base / 'cal'

patron_mic = re.compile(PLANTILLA_MICS, re.IGNORECASE)
patron_ref = re.compile(PLANTILLA_REF,  re.IGNORECASE)
patron_cal = re.compile(PLANTILLA_CAL,  re.IGNORECASE)

# ── Diccionarios de datos ──────────────────────────────────────────────────────
# mics[dinamica][angulo_mesa][mic_num] = señal float32
# refs[dinamica][angulo_mesa]          = señal float32
# cal[mic_num]                         = señal float32
mics = {}
refs = {}
cal  = {}
sr_global = None




## Carga de directorios

In [28]:
# ── Cargar micrófonos del array (carpeta media/) ───────────────────────────────
wavs_media = sorted(carpeta_media.rglob('*.wav'))
print(f"Archivos en media/: {len(wavs_media)}")

no_reconocidos_media = []
for wav in wavs_media:
    m = patron_mic.match(wav.name)
    if not m:
        no_reconocidos_media.append(wav.name)
        continue
    
    mic = int(m.group('MIC'))
    din = m.group('DIN').lower()
    ang = int(m.group('ANG'))
    
    sig, sr = sf.read(str(wav), dtype='float32')  # Carga de audios
    
    if sig.ndim > 1:
        sig = sig[:, 0]
    if sr_global is None:
        sr_global = sr
    mics.setdefault(din, {}).setdefault(ang, {})[mic] = sig

Archivos en media/: 722


In [29]:

# ── Cargar referencias (carpeta referencias/) ─────────────────────────────────────────
wavs_ref = sorted(carpeta_ref.rglob('*.wav'))
print(f"Archivos en ref/:   {len(wavs_ref)}")

no_reconocidos_ref = []
for wav in wavs_ref:
    m = patron_ref.match(wav.name)
    if not m:
        no_reconocidos_ref.append(wav.name)
        continue
    
    din = m.group('DIN').lower()
    ang = int(m.group('ANG'))
    sig, sr = sf.read(str(wav), dtype='float32')
    if sig.ndim > 1:
        sig = sig[:, 0]
    if sr_global is None:
        sr_global = sr
    refs.setdefault(din, {})[ang] = sig


Archivos en ref/:   38


In [30]:
# ── Cargar calibración (carpeta cal/) ─────────────────────────────────────────
wavs_cal = sorted(carpeta_cal.rglob('*.wav'))
print(f"Archivos en cal/:   {len(wavs_cal)}")

no_reconocidos_cal = []
for wav in wavs_cal:
    m = patron_cal.match(wav.name)
    if not m:
        no_reconocidos_cal.append(wav.name)
        continue
    mic = int(m.group('MIC'))
    sig, sr = sf.read(str(wav), dtype='float32')
    if sig.ndim > 1:
        sig = sig[:, 0]
    if sr_global is None:
        sr_global = sr
    cal[mic] = sig

Archivos en cal/:   20


In [31]:
# ── Resumen ────────────────────────────────────────────────────────────────────
print(f"Sample rate: {sr_global} Hz\n")

for din in sorted(mics.keys()):
    angulos = sorted(mics[din].keys())
    n_mics  = len(mics[din][angulos[0]]) if angulos else 0
    n_refs  = len(refs.get(din, {}))
    dur     = len(list(mics[din][angulos[0]].values())[0]) / sr_global if angulos else 0
    print(f"  {din.upper()}")
    print(f"    Ángulos de mesa:   {angulos}")
    print(f"    Micrófonos/ángulo: {n_mics}")
    print(f"    Referencias:       {n_refs}")
    print(f"    Duración aprox:    {dur:.2f} s")
    print()

print(f"  CALIBRACIÓN")
print(f"    Micrófonos cargados: {sorted(cal.keys())}")
dur_cal = len(cal[list(cal.keys())[0]]) / sr_global if cal else 0
print(f"    Duración aprox:      {dur_cal:.2f} s")
print()

if no_reconocidos_media:
    print(f"No reconocidos en media/ ({len(no_reconocidos_media)}):")
    for n in no_reconocidos_media: print(f"  {n}")

if no_reconocidos_ref:
    print(f"No reconocidos en ref/ ({len(no_reconocidos_ref)}):")
    for n in no_reconocidos_ref: print(f"  {n}")

if no_reconocidos_cal:
    print(f"No reconocidos en cal/ ({len(no_reconocidos_cal)}):")
    for n in no_reconocidos_cal: print(f"  {n}")

Sample rate: 44100 Hz

  FORTE
    Ángulos de mesa:   [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180]
    Micrófonos/ángulo: 19
    Referencias:       19
    Duración aprox:    14.02 s

  PIANO
    Ángulos de mesa:   [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180]
    Micrófonos/ángulo: 19
    Referencias:       19
    Duración aprox:    12.56 s

  CALIBRACIÓN
    Micrófonos cargados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
    Duración aprox:      14.07 s

No reconocidos en cal/ (1):
  mic_ref_ang_cal.wav


In [32]:
# Verificar shapes de un ejemplo de cada tipo
din = 'forte'
ang = 90
mic = 1

print(f"Medición  forte, mesa 90°, mic 1: {mics[din][ang][mic].shape} samples = {len(mics[din][ang][mic])/sr_global:.2f} s")
print(f"Referencia forte, mesa 90°:        {refs[din][ang].shape}")
print(f"Calibración mic 1:                 {cal[1].shape}")
print(f"Rango señal medición:  [{mics[din][ang][mic].min():.4f}, {mics[din][ang][mic].max():.4f}]")
print(f"Rango señal referencia: [{refs[din][ang].min():.4f}, {refs[din][ang].max():.4f}]")

Medición  forte, mesa 90°, mic 1: (599988,) samples = 13.61 s
Referencia forte, mesa 90°:        (272538,)
Calibración mic 1:                 (577460,)
Rango señal medición:  [-0.4359, 0.5386]
Rango señal referencia: [-0.5820, 0.6498]


In [33]:
# Detección de F0 robusta con pYIN
f0, voiced, prob = librosa.pyin(señal, fmin=300, fmax=750, sr=sr)

# STFT ya normalizada
D = librosa.stft(señal, n_fft=2048, hop_length=512)

# Conversión a dB
D_db = librosa.amplitude_to_db(np.abs(D))

# Filtros de bandas de 1/3 oct
freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

# Visualización integrada
librosa.display.specshow(D_db, sr=sr, x_axis='time', y_axis='log')

NameError: name 'señal' is not defined